In [ ]:
# Importando as Bibliotecas:
import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata
import pandas as pd


In [9]:
# Extraindo base com Chapas:
df_chapas = pd.read_excel(r'./base/PlanilhaChapa.xlsx')


In [10]:
# Inserindo base do ronda:
df_ronda = pd.read_parquet(r'./df_ronda.parquet')


In [11]:

df_chapas


,Cadastro,Nome,Empresa,Código do Local
0,9800441,DARIO JOSE APARECIDO LOPES,LEADEC SERVICOS INDUSTRIAIS DO BRAS ...,0000.010.5117.99.T
1,9812703,JOSE ARMANDO ANDRE JUNIOR,LEADEC SERVICOS INDUSTRIAIS DO BRAS ...,0000.010.5117.99.T
2,9799656,CLEUSA MESQUITA DA SILVA,LEADEC SERVICOS INDUSTRIAIS DO BRAS ...,0000.010.5117.99.T
3,9771289,BRASILINA TEIXEIRA,LEADEC SERVICOS INDUSTRIAIS DO BRAS ...,0000.010.5115.99.T
4,9771305,GERALDA VIEIRA DA SILVA RODRIGUES,LEADEC SERVICOS INDUSTRIAIS DO BRAS ...,0000.010.5115.99.T
...,...,...,...,...
708,9801443,CELIA MARIA DA SILVA,LEADEC SERVICOS INDUSTRIAIS DO BRAS ...,0000.013.1491.99.T
709,9784497,JOSE JUNIOR OLIVEIRA DE JESUS,LEADEC SERVICOS INDUSTRIAIS DO BRAS ...,0000.013.1448.99.T
710,9792352,JOSE HENRIQUE DE SOUZA,LEADEC SERVICOS INDUSTRIAIS DO BRAS ...,0000.013.1973.99.T
711,9806479,MARLI LOPES BARBOSA,LEADEC SERVICOS INDUSTRIAIS DO BRAS ...,0000.008.2362.99.T


In [ ]:
# Gerando output com indicador de registro no Ronda e percentual no universo de dias
pasta = Path('.')
saida = pasta / 'Output_Chapas_Existem_Registros.xlsx'

df_chapas = pd.read_excel(pasta / 'base' / 'PlanilhaChapa.xlsx')
df_ronda = pd.read_parquet(pasta / 'df_ronda.parquet')

df_chapas.columns = [str(col).strip() for col in df_chapas.columns]
df_ronda.columns = [str(col).strip().lower() for col in df_ronda.columns]

def normaliza_nome(valor):
    if pd.isna(valor):
        return pd.NA
    texto = str(valor).strip().upper()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    return ' '.join(texto.split())

nomes_planilha = df_chapas['Nome'].map(normaliza_nome).astype('string')
nomes_ronda = df_ronda['nome'].map(normaliza_nome).astype('string')

datas_ronda = pd.to_datetime(df_ronda['data'], format='%d/%m/%Y', errors='coerce').dt.normalize()
df_ronda_dias = pd.DataFrame({'nome': nomes_ronda, 'data': datas_ronda}).dropna()

total_registros_ronda = len(df_ronda)
total_dias_possiveis = df_ronda_dias['data'].nunique()
periodo_inicial = df_ronda_dias['data'].min()
periodo_final = df_ronda_dias['data'].max()
dias_corridos_periodo = (periodo_final - periodo_inicial).days + 1

dias_com_registro_por_nome = (
    df_ronda_dias.drop_duplicates(['nome', 'data'])
    .groupby('nome')['data']
    .size()
)

df_saida = df_chapas.copy()
df_saida['dias_com_registro'] = nomes_planilha.map(dias_com_registro_por_nome).fillna(0).astype(int)
df_saida['existem_registros'] = df_saida['dias_com_registro'].gt(0).map({True: 'Sim', False: 'Não'})
df_saida['percentual_universo_dias'] = df_saida['dias_com_registro'] / total_dias_possiveis

resumo = pd.DataFrame(
    [
        {'metrica': 'Total de chapas na planilha', 'valor': len(df_saida)},
        {'metrica': 'Nomes unicos na planilha', 'valor': nomes_planilha.nunique(dropna=True)},
        {'metrica': 'Chapas com registro no Ronda', 'valor': (df_saida['existem_registros'] == 'Sim').sum()},
        {'metrica': '% de chapas com registro no Ronda', 'valor': (df_saida['existem_registros'] == 'Sim').mean()},
        {'metrica': 'Chapas sem registro no Ronda', 'valor': (df_saida['existem_registros'] == 'Não').sum()},
        {'metrica': '% de chapas sem registro no Ronda', 'valor': (df_saida['existem_registros'] == 'Não').mean()},
        {'metrica': 'Total de registros no Ronda', 'valor': total_registros_ronda},
        {'metrica': 'Total de dias possiveis no Ronda', 'valor': total_dias_possiveis},
        {'metrica': 'Dias corridos no periodo', 'valor': dias_corridos_periodo},
        {'metrica': 'Periodo inicial do Ronda', 'valor': periodo_inicial.strftime('%d/%m/%Y')},
        {'metrica': 'Periodo final do Ronda', 'valor': periodo_final.strftime('%d/%m/%Y')},
        {'metrica': 'Media de dias com registro por chapa', 'valor': df_saida['dias_com_registro'].mean()},
        {'metrica': '% medio do universo de dias por chapa', 'valor': df_saida['percentual_universo_dias'].mean()},
        {'metrica': '% medio do universo de dias entre chapas com registro', 'valor': df_saida.loc[df_saida['existem_registros'] == 'Sim', 'percentual_universo_dias'].mean()},
    ]
)

with pd.ExcelWriter(saida, engine='openpyxl') as writer:
    df_saida.to_excel(writer, sheet_name='output', index=False)
    resumo.to_excel(writer, sheet_name='resumo', index=False)

    wb = writer.book
    ws_output = writer.sheets['output']
    ws_resumo = writer.sheets['resumo']

    for ws in (ws_output, ws_resumo):
        ws.freeze_panes = 'A2'
        for column_cells in ws.columns:
            max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
            ws.column_dimensions[column_cells[0].column_letter].width = min(max(max_len + 2, 12), 55)

    percentual_col = df_saida.columns.get_loc('percentual_universo_dias') + 1
    for cell in ws_output.iter_cols(min_col=percentual_col, max_col=percentual_col, min_row=2):
        for item in cell:
            item.number_format = '0.0000%'

    for row in ws_resumo.iter_rows(min_row=2, max_col=2):
        metrica = row[0].value or ''
        if str(metrica).startswith('%'):
            row[1].number_format = '0.00%'

print(f'Output gerado: {saida.resolve()}')
display(df_saida.head())
display(resumo)
